# Day 9 Project: Resume → Structured JSON Parser

## What You're Building

You run this notebook against a plain-text resume (included as a multi-line string).
The notebook extracts a fully validated `ResumeProfile` Pydantic model — personal
info, a list of `WorkExperience` entries, a list of `Education` entries, and a list
of skills — then pretty-prints the result as indented JSON. **That JSON is the
deliverable:** structured, typed, validated data produced entirely by a local Ollama
model with no paid API.

### Concepts composed from today's lessons

| Lesson | Concept used |
|--------|--------------|
| 1 | Schema-guided extraction (Define → Prompt → Validate) |
| 2 | Optional fields, nested models, `Field(description=...)` |
| 3 | Few-shot grounding — two example pairs injected before the real input |
| 4 | Extract-validate-retry loop — feed `ValidationError` back to the model |
| 5 | Section-aware extraction — split resume into sections, extract per section, assemble |

> Complete exercises 1–5 before starting this project. The patterns here build
> directly on what each exercise taught.

## Step 1 — Imports and configuration

In [ ]:
import json
import logging
import ollama
from pydantic import BaseModel, Field, ValidationError

# TODO: configure logging (level=DEBUG, format includes timestamp and level)
# TODO: set MODEL = "llama3.2"

MODEL = "llama3.2"

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s %(levelname)-8s %(message)s",
)
logger = logging.getLogger(__name__)

## Step 2 — Define the Pydantic schemas

You need four models:
- `PersonalInfo` — name (required), email (required), phone (optional), location (optional)
- `WorkExperience` — company, title, start_year, end_year (optional, null if current), responsibilities (list)
- `Education` — institution, degree, graduation_year (optional)
- `ResumeProfile` — personal (PersonalInfo), work (list of WorkExperience), education (list of Education), skills (list of str)

Give every field a `Field(description=...)` that tells the model what to extract and
what to return when the value is absent.

In [ ]:
# TODO: define PersonalInfo — name, email (required); phone, location (optional)
class PersonalInfo(BaseModel):
    pass


# TODO: define WorkExperience — company, title, start_year (required);
#       end_year (optional, null = current role); responsibilities (list[str])
class WorkExperience(BaseModel):
    pass


# TODO: define Education — institution, degree (required); graduation_year (optional)
class Education(BaseModel):
    pass


# TODO: define ResumeProfile — personal (PersonalInfo), work (list[WorkExperience]),
#       education (list[Education]), skills (list[str])
class ResumeProfile(BaseModel):
    pass

## Step 3 — The section splitter

Implement `split_sections(document, section_names)` — it scans the document
line by line, detects headers from `section_names` (case-insensitive), and
returns a `dict[str, str]` mapping section label to section text. Text before
the first recognised header goes under `"PREAMBLE"`.

In [ ]:
def split_sections(document: str, section_names: list[str]) -> dict[str, str]:
    """Split a document into labelled sections.

    Args:
        document:      Full document text.
        section_names: Section header names to detect (case-insensitive).

    Returns:
        dict mapping upper-cased section label to stripped section text.
        Text before the first header is stored under 'PREAMBLE'.
        Empty sections are omitted.
    """
    # TODO: normalise section_names to a set of upper-cased strings
    # TODO: iterate lines; detect headers, accumulate lines per section
    # TODO: join and strip each section; drop empty ones
    pass

## Step 4 — The extract-validate-retry function

Implement `extract_with_retry(text, model_class, max_retries)`. It should:
1. Build a system prompt embedding the model class's JSON Schema
2. Call `ollama.chat()` with `format="json"`
3. Validate with `model_class.model_validate_json(raw)`
4. On `ValidationError`, feed the error back as a user turn and retry
5. Raise the last `ValidationError` if all retries are exhausted

In [ ]:
def build_system_prompt(schema: dict) -> str:
    """Build the extraction system prompt embedding the given JSON Schema."""
    # TODO: return a prompt that instructs the model to extract into the schema,
    #       return only JSON, and set missing optional fields to null
    pass


def extract_with_retry(
    text: str,
    model_class: type[BaseModel],
    max_retries: int = 3,
) -> BaseModel:
    """Extract structured data from text, retrying on ValidationError.

    Args:
        text:        Source text to extract from.
        model_class: Pydantic BaseModel subclass defining the target schema.
        max_retries: Maximum correction attempts (default 3).

    Returns:
        A validated instance of model_class.

    Raises:
        ValidationError: If all attempts are exhausted without valid output.
    """
    # TODO: Step 1 — build schema and system prompt
    # TODO: Step 2 — initialise message list with system + user(text)
    # TODO: Step 3 — loop up to max_retries:
    #         call ollama.chat with format="json"
    #         validate with model_validate_json
    #         on success return result
    #         on ValidationError append failed output + error as repair turn
    # TODO: Step 4 — raise last_error after loop
    pass

## Step 5 — Few-shot examples

Write two `(input_prose, output_dict)` example pairs for a `ResumeProfile` extraction:
- Example 1: a short but complete resume snippet — all fields populated
- Example 2: a minimal snippet — only required fields, everything else `null` or empty

Then write `build_fewshot_messages(schema, examples, real_input)` that
assembles the full messages list (system + example pairs + real input).

In [ ]:
# TODO: write EXAMPLES — a list of two (str, dict) tuples
# Each dict must be a valid ResumeProfile JSON (use ResumeProfile.model_json_schema() to check keys)
EXAMPLES: list[tuple[str, dict]] = [
    # (example_prose_1, example_output_dict_1),
    # (example_prose_2, example_output_dict_2),
]


def build_fewshot_messages(
    schema: dict,
    examples: list[tuple[str, dict]],
    real_input: str,
) -> list[dict]:
    """Build the messages list: system + example pairs + real input.

    Args:
        schema:      JSON Schema dict from ResumeProfile.model_json_schema().
        examples:    List of (input_prose, output_dict) demonstration pairs.
        real_input:  The actual resume text to extract.

    Returns:
        A messages list ready to pass to ollama.chat().
    """
    # TODO: build system message with schema embedded
    # TODO: inject each example as a user/assistant pair
    # TODO: append the real input as the final user message
    pass

## Step 6 — The plain-text resume

This is your source document. Do not change the section headers (the extractor
looks for `SKILLS`, `EXPERIENCE`, and `EDUCATION` by name). You can edit the
content to test different extraction scenarios.

In [ ]:
RESUME_TEXT = """
Jordan Rivera
jordan.rivera@example.com
+1-312-555-0174
Chicago, IL

SKILLS
Python, SQL, Apache Spark, dbt, Airflow, data modelling, REST APIs, Docker

EXPERIENCE
Senior Data Engineer, Luminary Analytics, 2021-present.
Designed and maintained ETL pipelines processing 50M events/day using Python
and Apache Spark. Led migration from on-prem Hadoop to AWS EMR.

Data Engineer, GreenPath Technologies, 2018-2021.
Built dbt models for financial reporting. Wrote Airflow DAGs for nightly batch
jobs. Reduced pipeline failure rate by 40 percent through improved error handling.

Junior Analyst, DataFirst Consulting, 2016-2018.
Delivered SQL-based reports for retail clients. Automated weekly Excel
summaries with Python scripts.

EDUCATION
BSc Computer Science, University of Illinois at Chicago, 2016.
Certificate in Data Engineering, Coursera / Google, 2019.
"""

## Step 7 — Section-aware extraction pipeline

Implement `parse_resume(document)` that:
1. Calls `split_sections` to divide the resume into labelled parts
2. Extracts each section with the right schema using few-shot + retry
3. Assembles and returns one `ResumeProfile` instance

In [ ]:
def parse_resume(document: str) -> ResumeProfile:
    """Parse a plain-text resume into a validated ResumeProfile.

    Strategy:
      - Split the document into sections (PREAMBLE, SKILLS, EXPERIENCE, EDUCATION)
      - Extract PREAMBLE → PersonalInfo
      - Extract SKILLS section → list of skill strings
      - Extract EXPERIENCE section → list of WorkExperience objects
      - Extract EDUCATION section → list of Education objects
      - Assemble all parts into a ResumeProfile

    Args:
        document: Full plain-text resume string.

    Returns:
        A validated ResumeProfile instance.
    """
    # TODO: split the document
    # TODO: extract personal info from 'PREAMBLE'
    # TODO: extract skills from 'SKILLS' (hint: define a small SkillsSection schema)
    # TODO: extract work experience from 'EXPERIENCE' (hint: a WorkSection schema with list[WorkExperience])
    # TODO: extract education from 'EDUCATION' (hint: an EducationSection schema with list[Education])
    # TODO: assemble and return ResumeProfile
    pass

## Step 8 — Run it and print the deliverable

In [ ]:
# TODO: call parse_resume(RESUME_TEXT)
# TODO: print the result as indented JSON using model.model_dump() + json.dumps(..., indent=2)
# TODO: print a confirmation line at the end
pass